# Mapping vegetation greenness with HLS (Python)

**Question.** How green is the forest around the Pando aspen stand in midsummer, and which pixels can we trust?

By the end of this lesson you will have a map like this one, made from one clear satellite scene, and a saved file you can open in any GIS:

![Expected result: NDVI map of the Pando study area](figures/expected-ndvi-map.png)

**You will learn to**

1. search NASA's catalog for Harmonized Landsat Sentinel-2 (HLS) data and judge which scenes actually show your study area clearly;
2. read only the study-area part of cloud-hosted files, then apply scale factors and the Fmask quality layer;
3. map the Normalized Difference Vegetation Index (NDVI) and save it with a record of how it was made.

## Before starting

- **Setup.** Run the [getting-started notebook](../getting-started/) first. You need the Trees from Space Python environment and an Earthdata Login stored in `~/.netrc` ([guide](../../docs/getting-started/earthdata-login.md)).
- **CHPC.** OnDemand sessions on compute nodes cannot download data. Either fetch the data first with a short transfer job, or run on a Frisco node, as described in the [CHPC guide](../../docs/getting-started/chpc.md). Once the data are cached, the whole notebook runs offline.
- **Data volume.** About 2 MB, read directly from NASA's servers and saved to the lesson cache.
- **Time.** About 20 minutes of reading and running. The first run spends a few minutes reading data; reruns take under a minute.
- **Tested environments.** See the [lesson README](README.md).

## The data in brief

**HLS** (Harmonized Landsat Sentinel-2, version 2.0) combines observations from the Landsat 8 and 9 satellites (product **HLSL30**) and the Sentinel-2A, 2B, and 2C satellites (product **HLSS30**). NASA processes both to surface reflectance on the same 30 m grid and adjusts them so they can be used together. Together they observe most places every two to four days. Data are stored in tiles about 110 km across, and each file holds one spectral band of one tile on one date. A set of files from one overpass is called a **granule**.

**Surface reflectance** is the fraction of sunlight the ground reflects in a wavelength band, from 0 to 1. The files store it as whole numbers multiplied by 10,000, so a stored value of 3,512 means a reflectance of 0.3512.

**NDVI** compares near-infrared (NIR) and red reflectance: NDVI = (NIR − Red) / (NIR + Red). Green leaves reflect strongly in the near-infrared and absorb red light, so dense green vegetation has high NDVI (about 0.6 to 0.9), while bare soil is near 0.1 to 0.2 and water is usually below 0. NDVI has no units.

**Limits to keep in mind.** NDVI is a proxy for green leaf cover, not a measure of biomass. It saturates in dense canopies, so it separates sparse from dense forest well but cannot tell a dense stand from a very dense one. The Fmask quality layer flags most clouds and shadows, but thin cloud, smoke, and the edges of shadows can slip through.

## Settings

Everything you might change is in the next cell. Coordinates are longitude and latitude in decimal degrees (WGS84), in the order **west, south, east, north**. Dates use the format `"YYYY-MM-DD"`. If you run this notebook as a batch job, the job uses the values saved in this cell.

In [ ]:
# Study area: a box of longitude/latitude in degrees, ordered (west, south, east, north).
SITE_ID = "pando"
SITE_NAME = "Pando and Fish Lake, Utah"
BBOX = (-111.788, 38.495, -111.712, 38.555)

# Search window. Mid-July to mid-September avoids snow at this mountain site.
START_DATE = "2024-07-15"
END_DATE = "2024-09-15"

# HLS products (Landsat and Sentinel-2) and their version.
PRODUCTS = ["HLSL30", "HLSS30"]
VERSION = "2.0"

# Request limit: the most granules to consider per product.
MAX_GRANULES = 40

# Fmask bits that make a pixel unusable: 1 cloud, 2 adjacent to cloud or shadow,
# 3 cloud shadow, 4 snow or ice. (Bit 5, water, is shown on the map instead.)
MASK_BITS = [1, 2, 3, 4]

# The chosen scene must have at least this fraction of clear pixels in the study area.
MIN_CLEAR_FRACTION = 0.8

## Setup

The next cell loads the packages this lesson uses and sets up the folders for cached data and results. In Google Colab it first installs two packages, which takes a minute.

In [ ]:
import json
import os
import socket
import subprocess
import sys
import time
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    colab_packages = ["earthaccess==0.19.0", "rasterio==1.5.0"]
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *colab_packages], check=True)

import earthaccess
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from matplotlib.colors import ListedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from rasterio.io import MemoryFile
from rasterio.shutil import copy as copy_raster
from rasterio.warp import transform as transform_points
from rasterio.warp import transform_bounds
from rasterio.windows import Window

RUN_STARTED = time.monotonic()


def find_repo_root(start: Path) -> Path | None:
    """Walk up from `start` to the folder holding the .tfs-root marker file."""
    for folder in [start, *start.parents]:
        if (folder / ".tfs-root").exists():
            return folder
    return None


REPO_ROOT = find_repo_root(Path.cwd())
BASE_DIR = Path("/content/trees-from-space") if IN_COLAB else (REPO_ROOT or Path.cwd())
CACHE_DIR = Path(os.environ.get("TFS_CACHE_DIR", BASE_DIR / "cache"))
OUTPUT_DIR = Path(os.environ.get("TFS_OUTPUT_DIR", BASE_DIR / "outputs"))

# One cache folder per study box, named by its coordinates, so the R lesson finds the same files.
box_name = "_".join(f"{value:.4f}" for value in BBOX)
SITE_CACHE = CACHE_DIR / "hls" / f"{SITE_ID}_{box_name}"
LESSON_OUTPUT = OUTPUT_DIR / "hls-vegetation-map" / "python"
SITE_CACHE.mkdir(parents=True, exist_ok=True)
LESSON_OUTPUT.mkdir(parents=True, exist_ok=True)
print(f"Cache:   {SITE_CACHE}")
print(f"Results: {LESSON_OUTPUT}")

### Helpers for reading data once

Downloading the same data every time you rerun a notebook wastes time, and on CHPC compute nodes it is not even possible. The next cell defines a few small functions that save every piece of downloaded data in the cache and reuse it on later runs. Each file is first written under a temporary name and renamed only when complete, so an interrupted run never leaves a half-written file that looks finished.

The settings for GDAL, the library that reads the satellite files, tell it to use your `~/.netrc` credentials, to keep NASA's login cookies in a private folder, and to retry a few times before giving up.

In [ ]:
NETRC_PATH = Path(os.environ.get("NETRC", Path.home() / ".netrc"))
COOKIE_DIR = Path.home() / ".cache" / "trees-from-space"
COOKIE_DIR.mkdir(parents=True, exist_ok=True)
COOKIE_DIR.chmod(0o700)
GDAL_OPTIONS = {
    "GDAL_HTTP_NETRC": "YES",
    "GDAL_HTTP_NETRC_FILE": str(NETRC_PATH),
    "GDAL_HTTP_COOKIEFILE": str(COOKIE_DIR / "earthdata-cookies.txt"),
    "GDAL_HTTP_COOKIEJAR": str(COOKIE_DIR / "earthdata-cookies.txt"),
    "GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR",
    "CPL_VSIL_CURL_ALLOWED_EXTENSIONS": ".tif",
    "GDAL_HTTP_MAX_RETRY": "4",
    "GDAL_HTTP_RETRY_DELAY": "2",
    "GDAL_HTTP_CONNECTTIMEOUT": "10",
    "GDAL_HTTP_TIMEOUT": "60",
}


def network_available(host: str = "cmr.earthdata.nasa.gov", timeout_s: float = 5) -> bool:
    """Return True if this machine can open a connection to NASA."""
    if os.environ.get("TFS_OFFLINE") == "1":  # lets you test the offline route
        return False
    try:
        with socket.create_connection((host, 443), timeout=timeout_s):
            return True
    except OSError:
        return False


def offline_message(missing: Path) -> str:
    return (
        f"This machine cannot reach NASA, and the cache does not yet hold {missing.name}. "
        "Run the notebook once on a machine with internet access (on CHPC, a transfer job "
        "or a Frisco node; see docs/getting-started/chpc.md), then rerun it here."
    )


def write_atomically(path: Path, write) -> None:
    """Call write(temporary_path), then rename the result to `path`."""
    temporary = path.with_name(path.name + ".part")
    write(temporary)
    os.replace(temporary, path)


login_state = {"done": False}


def ensure_login() -> None:
    """Log in to Earthdata once, just before the first download."""
    if login_state["done"]:
        return
    if IN_COLAB and not NETRC_PATH.exists():
        try:  # Colab Secrets panel, if you added EARTHDATA_USERNAME and EARTHDATA_PASSWORD
            from google.colab import userdata

            os.environ["EARTHDATA_USERNAME"] = userdata.get("EARTHDATA_USERNAME")
            os.environ["EARTHDATA_PASSWORD"] = userdata.get("EARTHDATA_PASSWORD")
            earthaccess.login(strategy="environment", persist=True)
        except Exception:
            earthaccess.login(strategy="interactive", persist=True)
    if not NETRC_PATH.exists():
        raise RuntimeError(
            f"No Earthdata credentials at {NETRC_PATH}. "
            "See docs/getting-started/earthdata-login.md."
        )
    if not earthaccess.login(strategy="netrc").authenticated:
        raise RuntimeError("NASA did not accept the credentials in ~/.netrc.")
    login_state["done"] = True


ONLINE = network_available()
print("Network: online" if ONLINE else "Network: offline (only cached data can be used)")

## 1. Find granules and inspect the results

NASA's Common Metadata Repository (CMR) is the catalog of NASA Earth science data. The `earthaccess` package searches it by product name, version, area, and dates. Searching does not need a login. The search returns one record per granule, with links to each band file and a few properties, including the **tile-wide** cloud cover that NASA computed for the whole 110 km tile.

The next cell keeps the properties we need from each record. The granule ID encodes the product, tile, year, day of year, and time in UTC: `HLS.S30.T12SVH.2024197T181919.v2.0` is a Sentinel-2 granule of tile 12SVH from day 197 of 2024 (July 15) at 18:19:19 UTC. For Landsat (L30) the near-infrared band is `B05`, and for Sentinel-2 (S30) it is `B8A`. HLS uses Sentinel-2's narrow near-infrared band because it matches Landsat's.

In [ ]:
BANDS = {"HLSL30": {"red": "B04", "nir": "B05"}, "HLSS30": {"red": "B04", "nir": "B8A"}}
FILES_NEEDED = ["Fmask", "red", "nir"]


def granule_record(granule, product: str) -> dict:
    """Keep the properties of one search result that this lesson uses."""
    umm = granule["umm"]
    attributes = {a["Name"]: a["Values"][0] for a in umm.get("AdditionalAttributes", [])}
    links = {
        url.split(".")[-2]: url  # the band name sits just before ".tif"
        for url in granule.data_links(access="external")
        if url.startswith("https://") and url.endswith(".tif")
    }
    archive = umm.get("DataGranule", {}).get("ArchiveAndDistributionInformation", [])
    bands = {"Fmask": "Fmask", **BANDS[product]}
    return {
        "granule_id": umm["GranuleUR"],
        "product": product,
        "datetime_utc": umm["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"],
        "tile": umm["GranuleUR"].split(".")[2],
        "tile_cloud_pct": float(attributes["CLOUD_COVERAGE"]),
        "tile_data_pct": float(attributes["SPATIAL_COVERAGE"]),
        "urls": {name: links[band] for name, band in bands.items()},
        "granule_mb": sum(f.get("SizeInBytes", 0) for f in archive) / 1e6,  # all files together
    }


def load_or_search(product: str) -> list[dict]:
    """Return search results for one product, from the cache when possible."""
    path = SITE_CACHE / f"search_cmr_{product}_v{VERSION}_{START_DATE}_{END_DATE}_max{MAX_GRANULES}.json"
    if path.exists():
        print(f"{product}: {path.name} (cached)")
        return json.loads(path.read_text())
    if not ONLINE:
        raise RuntimeError(offline_message(path))
    results = earthaccess.search_data(
        short_name=product,
        version=VERSION,
        bounding_box=BBOX,
        temporal=(f"{START_DATE}T00:00:00Z", f"{END_DATE}T23:59:59Z"),
        count=MAX_GRANULES,
    )
    records = [granule_record(granule, product) for granule in results]
    write_atomically(path, lambda tmp: tmp.write_text(json.dumps(records, indent=1)))
    print(f"{product}: {len(records)} granules found (searched NASA's catalog)")
    return records


records = [record for product in PRODUCTS for record in load_or_search(product)]
if not records:
    raise RuntimeError(
        "No HLS granules matched. Check that BBOX is ordered (west, south, east, north), "
        "that START_DATE comes before END_DATE, and that the dates fall within the HLS "
        "record (Landsat from April 2013, Sentinel-2 from November 2015)."
    )

granules = pd.DataFrame(records)
granules["date"] = pd.to_datetime(granules["datetime_utc"]).dt.date
granules[["granule_id", "product", "date", "tile", "tile_cloud_pct", "tile_data_pct"]]

The table shows every granule found: its product, date, tile, the percentage of the whole tile that NASA flagged as cloud or cloud shadow (`tile_cloud_pct`), and the percentage of the tile with data (`tile_data_pct`). The catalog matched these granules because their **footprints** overlap our box. That says nothing yet about whether our box is clear, or even covered, in each one.

HLS tiles overlap their neighbors by about 10 km, so a study area near a tile edge can appear in two tiles with slightly different grids. Using a single tile keeps every image on the same grid. The next cell keeps the most common tile and reports how much data the next steps will read.

In [ ]:
tiles = granules["tile"].value_counts()
TILE = tiles.index[0]
if len(tiles) > 1:
    print(f"The box touches tiles {', '.join(tiles.index)}; using {TILE} only.")
granules = granules[granules["tile"] == TILE].reset_index(drop=True)
print(f"{len(granules)} granules from tile {TILE}: "
      f"{(granules['product'] == 'HLSL30').sum()} Landsat (L30), "
      f"{(granules['product'] == 'HLSS30').sum()} Sentinel-2 (S30)")

print(f"Each granule's files total about {granules['granule_mb'].median():.0f} MB, "
      f"or {granules['granule_mb'].sum() / 1e3:.1f} GB for all {len(granules)} granules.")
print("This lesson needs only three files per granule at most (Fmask, red, near-infrared),")
print("and from each only the small window that covers the study box.")

## 2. Check the study area in each granule

Each HLS granule includes an **Fmask** quality layer: one byte per pixel, in which individual bits flag cloud, cloud shadow, snow, water, and more. Bit 0 is the rightmost bit. For example, the value 66 is `01000010` in binary: bit 1 (cloud) is set, and bits 6 and 7 hold an aerosol level that we do not use here. A value of 255 means there is no data for that pixel.

| Bit | Meaning | Used here |
| --- | --- | --- |
| 1 | Cloud | Masked |
| 2 | Adjacent to cloud or shadow | Masked |
| 3 | Cloud shadow | Masked |
| 4 | Snow or ice | Masked |
| 5 | Water | Shown on the map |
| 6–7 | Aerosol level | Not used |

The files are **Cloud Optimized GeoTIFFs**, stored in blocks so that a program can read part of a file without downloading all of it. The next cell works out which pixels of the tile cover our box, reads that window of each granule's Fmask layer, and computes the **clear fraction**: the share of the box's pixels that have data and none of the masked flags. Pixels without data count against a granule.

The box is defined in longitude and latitude, but HLS files use a UTM map projection in meters. The window therefore covers the box's footprint in UTM, rounded outward to whole 30 m pixels, so it is slightly larger than the box itself.

In [ ]:
FMASK_FILL = 255
WATER_BIT = 5


def aoi_window(src) -> Window:
    """Pixel window covering BBOX in the file's grid, rounded outward to whole pixels."""
    left, bottom, right, top = transform_bounds("EPSG:4326", src.crs, *BBOX, densify_pts=21)
    col_left, row_top = ~src.transform * (left, top)
    col_right, row_bottom = ~src.transform * (right, bottom)
    col0, row0 = int(np.floor(col_left)), int(np.floor(row_top))
    col1, row1 = int(np.ceil(col_right)), int(np.ceil(row_bottom))
    return Window(col0, row0, col1 - col0, row1 - row0)


def read_aoi_window(url: str) -> tuple[np.ndarray, dict, str]:
    """Return (pixels, metadata, status) for the study-area window of one remote file."""
    path = SITE_CACHE / Path(url).name.replace(".tif", ".aoi.tif")
    if not path.exists():
        if not ONLINE:
            raise RuntimeError(offline_message(path))
        ensure_login()
        with rasterio.Env(**GDAL_OPTIONS), rasterio.open(f"/vsicurl/{url}") as src:
            window = aoi_window(src)
            pixels = src.read(1, window=window)
            profile = {
                "driver": "GTiff", "dtype": src.dtypes[0], "count": 1, "crs": src.crs,
                "width": window.width, "height": window.height, "nodata": src.nodata,
                "transform": src.window_transform(window), "compress": "deflate",
            }
            scales, offsets, tags = src.scales, src.offsets, src.tags()

        def write(tmp: Path) -> None:
            with rasterio.open(tmp, "w", **profile) as dst:
                dst.write(pixels, 1)
                dst.scales, dst.offsets = scales, offsets
                dst.update_tags(**tags)

        write_atomically(path, write)
        status = "downloaded"
    else:
        status = "cached"
    with rasterio.open(path) as src:
        meta = {"crs": src.crs, "transform": src.transform, "nodata": src.nodata,
                "scale": src.scales[0], "offset": src.offsets[0], "tags": src.tags(), "path": path}
        return src.read(1), meta, status


def fmask_bit(fmask: np.ndarray, bit: int) -> np.ndarray:
    """True where the given bit of the Fmask byte is set."""
    return ((fmask >> bit) & 1).astype(bool)


def fmask_classes(fmask: np.ndarray) -> dict[str, np.ndarray]:
    """Split pixels into no data, masked, water, and clear."""
    no_data = fmask == FMASK_FILL
    masked = np.zeros(fmask.shape, dtype=bool)
    for bit in MASK_BITS:
        masked |= fmask_bit(fmask, bit)
    masked &= ~no_data
    clear = ~no_data & ~masked
    water = clear & fmask_bit(fmask, WATER_BIT)
    return {"no_data": no_data, "masked": masked, "water": water, "clear": clear}


statuses, rows = [], []
for granule in granules.itertuples():
    fmask, fmask_meta, status = read_aoi_window(granule.urls["Fmask"])
    statuses.append(status)
    classes = fmask_classes(fmask)
    rows.append({
        "granule_id": granule.granule_id,
        "aoi_clear_pct": 100 * int(classes["clear"].sum()) / fmask.size,
        "aoi_data_pct": 100 * int((~classes["no_data"]).sum()) / fmask.size,
    })
print(f"Fmask windows: {statuses.count('downloaded')} downloaded, {statuses.count('cached')} cached")
window_kb = fmask.size / 1e3  # one byte per Fmask pixel, before compression
print(f"Window: {fmask.shape[1]} x {fmask.shape[0]} pixels of 30 m, about {window_kb:.0f} KB per Fmask "
      f"window and {2 * window_kb:.0f} KB per reflectance band before compression.")
print(f"In total this lesson reads about {(len(granules) + 4) * window_kb / 1e3:.1f} MB, "
      f"instead of {granules['granule_mb'].sum() / 1e3:.1f} GB for the full granules.")

granules = granules.merge(pd.DataFrame(rows), on="granule_id")

Now rank the granules. The best granule has the highest clear fraction in the study area. Ties are broken by the share of the box with any data, then by the date closest to the middle of the search window, then by granule ID, so the choice is always the same.

The plot compares NASA's tile-wide cloud estimate with what we measured inside the box. If the box were typical of the tile, the points would fall along the dashed line. Points far from it show why it pays to check the study area itself.

In [ ]:
start, end = date.fromisoformat(START_DATE), date.fromisoformat(END_DATE)
middle = start + timedelta(days=(end - start).days // 2)
granules["days_from_middle"] = [abs((d - middle).days) for d in granules["date"]]
granules = granules.sort_values(
    ["aoi_clear_pct", "aoi_data_pct", "days_from_middle", "granule_id"],
    ascending=[False, False, True, True],
).reset_index(drop=True)
best = granules.iloc[0]

if best["aoi_clear_pct"] < 100 * MIN_CLEAR_FRACTION:
    raise RuntimeError(
        f"No granule has at least {MIN_CLEAR_FRACTION:.0%} clear pixels in the study area "
        f"(best: {best['aoi_clear_pct']:.0f}%). Widen the date window or lower MIN_CLEAR_FRACTION."
    )

STYLE = {"HLSL30": ("#2a78d6", "o", "Landsat (L30)"), "HLSS30": ("#eb6834", "s", "Sentinel-2 (S30)")}
fig, ax = plt.subplots(figsize=(6.5, 4.5), layout="constrained")
ax.plot([0, 100], [100, 0], ls="--", lw=1, color="#8a8984", zorder=1)
ax.text(62, 41, "box as cloudy as the tile", color="#52514e", fontsize=8, rotation=-33)
for product, (color, marker, label) in STYLE.items():
    subset = granules[granules["product"] == product]
    ax.scatter(subset["tile_cloud_pct"], subset["aoi_clear_pct"], s=36, color=color,
               marker=marker, edgecolors="white", linewidths=0.8, label=label, zorder=3)
ax.scatter(best["tile_cloud_pct"], best["aoi_clear_pct"], s=140, facecolors="none",
           edgecolors="#0b0b0b", linewidths=1.2, zorder=4)
ax.annotate(f"selected: {best['date']}", (best["tile_cloud_pct"], best["aoi_clear_pct"]),
            xytext=(10, -14), textcoords="offset points", fontsize=8, color="#0b0b0b")
ax.set(xlim=(-3, 103), ylim=(-3, 103), xlabel="Tile-wide cloud and shadow (%, from NASA metadata)",
       ylabel="Clear pixels in the study area (%)")
ax.set_title(f"Which granules show the study area clearly? ({START_DATE} to {END_DATE})", fontsize=10)
ax.grid(color="#e4e3df", lw=0.6)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, fontsize=8, loc="lower left")
plt.show()

granules[["granule_id", "product", "date", "tile_cloud_pct", "aoi_clear_pct", "aoi_data_pct"]].head(8)

## 3. Read and prepare the selected granule

Now read the red and near-infrared windows of the selected granule. Before computing NDVI, convert the stored integers to reflectance. Each file records its own **scale factor** and **offset** (reflectance = stored value × scale + offset), and GDAL reports them for every band. The lesson uses those values rather than assuming them, because a few HLS files store the scale factor in an unexpected place. Stored values of −9999 mean "no observation" and become missing values (`NaN`, "not a number").

In [ ]:
HLS_SCALE = 0.0001  # documented in the HLS v2.0 user guide


def to_reflectance(pixels: np.ndarray, meta: dict, name: str) -> np.ndarray:
    """Convert stored integers to reflectance, with missing values as NaN."""
    scale, offset = meta["scale"], meta["offset"]
    if scale == 1.0:  # the file carries no scale factor
        print(f"Note: {name} has no scale factor in its metadata; using {HLS_SCALE}.")
        scale, offset = HLS_SCALE, 0.0
    reflectance = pixels.astype("float64") * scale + offset
    reflectance[pixels == meta["nodata"]] = np.nan
    return reflectance


fmask, fmask_meta, _ = read_aoi_window(best["urls"]["Fmask"])
red_pixels, red_meta, red_status = read_aoi_window(best["urls"]["red"])
nir_pixels, nir_meta, nir_status = read_aoi_window(best["urls"]["nir"])
statuses += [red_status, nir_status]
assert red_meta["transform"] == nir_meta["transform"] == fmask_meta["transform"], "grids differ"

red = to_reflectance(red_pixels, red_meta, "red")
nir = to_reflectance(nir_pixels, nir_meta, "near-infrared")
classes = fmask_classes(fmask)

print(f"Selected: {best['granule_id']} ({best['product']}, {best['date']})")
print(f"Red and NIR windows: {red_status}, {nir_status}")
print(f"Scale and offset used: red {red_meta['scale']} / {red_meta['offset']}, "
      f"NIR {nir_meta['scale']} / {nir_meta['offset']}")
print(f"Red reflectance of clear pixels: {np.nanmin(red[classes['clear']]):.3f} to {np.nanmax(red[classes['clear']]):.3f}")
print(f"NIR reflectance of clear pixels: {np.nanmin(nir[classes['clear']]):.3f} to {np.nanmax(nir[classes['clear']]):.3f}")

Reflectance should fall between 0 and 1. A few values slightly below 0 can occur over dark surfaces, such as water and deep shadow, because atmospheric correction is not perfect. If you see values in the thousands, the scale factor was not applied.

## 4. Compute and map NDVI

NDVI is computed only for clear pixels. Masked pixels and pixels without data become `NaN`, and so does any pixel where NIR + Red is zero, to avoid dividing by zero. Water keeps its NDVI value but is drawn in blue on the map and left out of the summary, which describes the land.

In [ ]:
with np.errstate(divide="ignore", invalid="ignore"):
    ndvi = (nir - red) / (nir + red)
ndvi[(nir + red) == 0] = np.nan
ndvi[~classes["clear"]] = np.nan

land = np.isfinite(ndvi) & ~classes["water"]
summary = {
    "site_id": SITE_ID,
    "granule_id": best["granule_id"],
    "product": best["product"],
    "date": str(best["date"]),
    "aoi_pixels": int(ndvi.size),
    "clear_land_pixels": int(land.sum()),
    "water_pixels": int(classes["water"].sum()),
    "masked_pixels": int(classes["masked"].sum()),
    "no_data_pixels": int(classes["no_data"].sum()),
    "clear_fraction": round(int(classes["clear"].sum()) / ndvi.size, 4),
    "ndvi_median": round(float(np.median(ndvi[land])), 4),
    "ndvi_p25": round(float(np.percentile(ndvi[land], 25)), 4),
    "ndvi_p75": round(float(np.percentile(ndvi[land], 75)), 4),
}
pd.Series(summary)

In [ ]:
transform = red_meta["transform"]
rows_n, cols_n = ndvi.shape
extent = (transform.c, transform.c + cols_n * transform.a, transform.f + rows_n * transform.e, transform.f)

# The study box, densified and converted from longitude/latitude to the map's UTM coordinates.
west, south, east, north = BBOX
steps = np.linspace(0, 1, 50)
box_lon = np.concatenate([west + (east - west) * steps, np.full(50, east),
                          east - (east - west) * steps, np.full(50, west)])
box_lat = np.concatenate([np.full(50, south), south + (north - south) * steps,
                          np.full(50, north), north - (north - south) * steps])
box_x, box_y = transform_points("EPSG:4326", red_meta["crs"], box_lon, box_lat)

# Categories drawn on top of NDVI: 0 water, 1 masked by Fmask.
overlay = np.full(ndvi.shape, np.nan)
overlay[classes["water"]] = 0
overlay[classes["masked"]] = 1
WATER_COLOR, MASKED_COLOR = "#2a78d6", "#b5b4ae"

fig, ax = plt.subplots(figsize=(7, 6.2), layout="constrained")
image = ax.imshow(np.where(land, ndvi, np.nan), extent=extent, cmap="Greens", vmin=0, vmax=0.9,
                  interpolation="nearest")
ax.imshow(overlay, extent=extent, cmap=ListedColormap([WATER_COLOR, MASKED_COLOR]),
          vmin=-0.5, vmax=1.5, interpolation="nearest")
ax.plot(box_x, box_y, color="#0b0b0b", lw=1.2)
# Approximate center of the Pando aspen clone (only meaningful for the example site).
if SITE_ID == "pando":
    (pando_x,), (pando_y,) = transform_points("EPSG:4326", red_meta["crs"], [-111.7503], [38.5247])
    ax.plot(pando_x, pando_y, marker="^", ms=8, color="#0b0b0b", mec="white", mew=0.8)
    ax.annotate("Pando clone (approx.)", (pando_x, pando_y), xytext=(8, 6),
                textcoords="offset points", fontsize=8, color="#0b0b0b")
fig.colorbar(image, ax=ax, shrink=0.8, label="NDVI (unitless)")
ax.legend(handles=[Patch(color=WATER_COLOR, label="Water (Fmask)"),
                   Patch(color=MASKED_COLOR, label="Cloud, shadow, or snow (Fmask)"),
                   Line2D([], [], color="#0b0b0b", lw=1.2, label="Study box")],
          loc="lower left", fontsize=8, framealpha=0.9)
sensor = "Landsat" if best["product"] == "HLSL30" else "Sentinel-2"
ax.set_title(f"NDVI, {SITE_NAME}: HLS {sensor}, {best['date']}\n"
             f"median {summary['ndvi_median']:.2f} over {summary['clear_land_pixels']:,} clear land pixels (30 m)",
             fontsize=10)
ax.set_xlabel(f"Easting (m), {red_meta['crs'].to_string()}")
ax.set_ylabel("Northing (m)")
ax.ticklabel_format(style="plain", useOffset=False)
ax.tick_params(labelsize=8)
plt.show()

**Reading the map.** Dark green areas are dense, leafy vegetation, mostly aspen and conifer forest. Lighter greens are meadows, shrubland, and sparse or stressed vegetation. Fish Lake fills the eastern edge in blue, and a few small ponds show as blue specks. Near-white pixels have NDVI close to or below zero, such as the wet lakeshore. The triangle marks the approximate center of the Pando clone.

Remember that NDVI saturates over dense canopy, so similar dark greens can hide real differences in forest structure. Later lessons on lidar and GEDI measure structure directly. NDVI also depends on the date: a scene from late August shows a canopy near the end of summer, before aspen leaves turn.

## 5. Save the results

Save the NDVI map as a Cloud Optimized GeoTIFF that any GIS can open, plus the figure, a one-row summary table, the granule ranking, and a **provenance** file that records exactly which data and settings produced the result. The provenance file holds the data links, but no credentials.

In [ ]:
def write_cog(path: Path, array: np.ndarray, meta: dict) -> None:
    """Write a single-band float32 Cloud Optimized GeoTIFF with NaN as no-data."""
    profile = {"driver": "GTiff", "dtype": "float32", "count": 1, "width": array.shape[1],
               "height": array.shape[0], "crs": meta["crs"], "transform": meta["transform"],
               "nodata": np.nan}

    def write(tmp: Path) -> None:
        with MemoryFile() as memory, memory.open(**profile) as staging:
            staging.write(array.astype("float32"), 1)
            staging.set_band_description(1, "NDVI (unitless)")
            copy_raster(staging, tmp, driver="COG", compress="DEFLATE", predictor="YES")

    write_atomically(path, write)


stem = f"{SITE_ID}_{best['date']}_ndvi"
write_cog(LESSON_OUTPUT / f"{stem}.tif", ndvi, red_meta)
fig.savefig(LESSON_OUTPUT / f"{stem}.png", dpi=150)
pd.DataFrame([summary]).to_csv(LESSON_OUTPUT / f"{stem}_summary.csv", index=False)
granules.drop(columns=["urls"]).to_csv(LESSON_OUTPUT / f"{SITE_ID}_granule_ranking.csv", index=False)

provenance = {
    "lesson": "tutorials/hls-vegetation-map",
    "language": "python",
    "created_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "settings": {"site_id": SITE_ID, "bbox_wsen_deg": list(BBOX), "start_date": START_DATE,
                 "end_date": END_DATE, "products": PRODUCTS, "version": VERSION,
                 "max_granules": MAX_GRANULES, "mask_bits": MASK_BITS,
                 "min_clear_fraction": MIN_CLEAR_FRACTION},
    "granules_considered": granules["granule_id"].tolist(),
    "selected_granule": {
        "granule_id": best["granule_id"], "product": best["product"],
        "datetime_utc": best["datetime_utc"], "tile": TILE,
        "sensing_time": red_meta["tags"].get("SENSING_TIME"),
        "tile_cloud_pct": best["tile_cloud_pct"], "aoi_clear_pct": round(best["aoi_clear_pct"], 2),
        "source_urls": best["urls"],
    },
    "processing": {
        "reflectance": "stored value x scale + offset, read from each file; -9999 -> NaN",
        "scale_offset": {"red": [red_meta["scale"], red_meta["offset"]],
                         "nir": [nir_meta["scale"], nir_meta["offset"]]},
        "ndvi": "(NIR - Red) / (NIR + Red) for clear pixels; NaN where masked, no data, or NIR + Red = 0",
        "fmask_masked_bits": MASK_BITS, "fmask_water_bit": WATER_BIT, "fmask_fill": FMASK_FILL,
        "grid": {"crs": red_meta["crs"].to_string(), "transform": list(transform)[:6],
                 "shape_rows_cols": list(ndvi.shape)},
        "summary_excludes": "water pixels",
    },
    "software": {"python": sys.version.split()[0], "earthaccess": earthaccess.__version__,
                 "rasterio": rasterio.__version__, "gdal": rasterio.__gdal_version__,
                 "numpy": np.__version__},
    "data_citation": [
        "Masek, J. et al. HLS Operational Land Imager Surface Reflectance and TOA Brightness Daily Global 30m v2.0. NASA EOSDIS LP DAAC. doi:10.5067/HLS/HLSL30.002",
        "Masek, J. et al. HLS Sentinel-2 Multi-spectral Instrument Surface Reflectance Daily Global 30m v2.0. NASA EOSDIS LP DAAC. doi:10.5067/HLS/HLSS30.002",
    ],
}
(LESSON_OUTPUT / f"{stem}_provenance.json").write_text(json.dumps(provenance, indent=1) + "\n")
for path in sorted(LESSON_OUTPUT.glob(f"{SITE_ID}*")):
    print(f"Saved {path.name}")

## Adapt it

**Exercise.** Map a forest you study. Change `SITE_ID`, `SITE_NAME`, and `BBOX` in the settings cell to a box of a few kilometers (larger boxes take longer to read), choose growing-season dates, and run the whole notebook again (*Restart Kernel and Run All Cells*). Does the clearest scene come from Landsat or Sentinel-2? How different are the tile-wide cloud cover and the clear fraction in your box?

**Going further**

- [HLS v2.0 user guide](https://lpdaac.usgs.gov/documents/1698/HLS_User_Guide_V2.pdf): processing, bands, and the Fmask bits.
- NASA also publishes ready-made vegetation indices from HLS (products `HLSL30_VI` and `HLSS30_VI`). They are convenient once you understand the masking and scaling this lesson teaches.
- The [NASA HLS tutorials](https://github.com/nasa/HLS-Data-Resources) show time series and larger areas.
- The next lessons read Sentinel-2 imagery from a STAC catalog and map canopy height from GEDI lidar.

**Data citation.** Masek, J. et al. (2021). HLS Operational Land Imager Surface Reflectance and TOA Brightness Daily Global 30m v2.0 ([doi:10.5067/HLS/HLSL30.002](https://doi.org/10.5067/HLS/HLSL30.002)) and HLS Sentinel-2 Multi-spectral Instrument Surface Reflectance Daily Global 30m v2.0 ([doi:10.5067/HLS/HLSS30.002](https://doi.org/10.5067/HLS/HLSS30.002)). NASA EOSDIS Land Processes Distributed Active Archive Center.

## Run summary

The last cell prints a short summary of this run. If you are asked to report a test of this lesson, copy everything it prints.

In [ ]:
def detect_platform() -> str:
    """Name the kind of machine running this notebook (no hostnames are reported)."""
    if IN_COLAB:
        return "colab"
    on_chpc = "chpc.utah.edu" in socket.getfqdn() or Path("/scratch/general/vast").is_dir()
    if not on_chpc:
        return "local"
    if socket.gethostname().startswith("frisco"):
        return "chpc-frisco"
    partition = os.environ.get("SLURM_JOB_PARTITION")
    return f"chpc-{partition}" if partition else "chpc-login"


run_summary = {
    "lesson": "hls-vegetation-map",
    "language": "python",
    "platform": detect_platform(),
    "run_utc": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "network": ONLINE,
    "files_downloaded": statuses.count("downloaded"),
    "files_cached": statuses.count("cached"),
    "granules_considered": len(granules),
    **{key: summary[key] for key in ("granule_id", "aoi_pixels", "clear_land_pixels", "clear_fraction",
                                     "ndvi_median", "ndvi_p25", "ndvi_p75")},
    "elapsed_s": round(time.monotonic() - RUN_STARTED, 1),
    "versions": provenance["software"],
}
print(json.dumps(run_summary, indent=1))